# **Laboratorio 04**

In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_codigo.csv')

**Consolidación y limpieza de datos**

A.

In [4]:

# Leer los archivos
lista_df = [pd.read_csv(archivo) for archivo in archivos_anio]

# Convertir los nombres de las columnas a minúsculas
for df_temp in lista_df:
    df_temp.columns = df_temp.columns.str.lower()

# Unir ambos DataFrames
df_anio = pd.concat(lista_df, ignore_index=True)

# Mostrar las primeras filas
df_anio.head()

,codigo_iso,anio,indice,ranking
0,AFG,2001,35.5,59.0
1,AGO,2001,30.2,50.0
2,ALB,2001,NaN,NaN
3,AND,2001,NaN,NaN
4,ARE,2001,NaN,NaN


B.

In [6]:
# Buscar el código ISO duplicado
df_codigos[df_codigos.duplicated(subset="codigo_iso", keep=False)]

# Eliminar el registro incorrecto
df_codigos = df_codigos[df_codigos["pais"] != "malo"]

# Verificar que ya no existan códigos ISO duplicados
df_codigos[df_codigos.duplicated(subset="codigo_iso", keep=False)]

,codigo_iso,pais


C.

In [7]:
# Combinar ambos DataFrames mediante codigo_iso
df = pd.merge(
    df_anio,
    df_codigos,
    on="codigo_iso",
    how="inner"
)

# Mostrar las primeras filas
df.head()

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos


**Exploración inicial del conjunto de datos**


In [8]:
#¿Cuántas filas y columnas tiene?
df.shape
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

#¿Cuáles son los nombres de las columnas?
print(df.columns.tolist())

#¿Qué tipo de datos tiene cada columna?
df.dtypes



Filas: 3060
Columnas: 5
['codigo_iso', 'anio', 'indice', 'ranking', 'pais']


,0
codigo_iso,object
anio,int64
indice,float64
ranking,float64
pais,object


**¿Hay columnas con un tipo de dato inesperado?**

No.

In [13]:
#Resumen estadistico
df.describe()


,anio,indice,ranking
count,3060.000000,2664.000000,2837.000000
mean,2009.941176,205.782316,477.930913
std,5.786024,2695.525264,6474.935347
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.295000,34.000000
50%,2009.000000,28.000000,70.000000
75%,2015.000000,41.227500,110.000000
max,2019.000000,64536.000000,121056.000000


**¿Qué observas sobre los valores de índice y ranking?**

Los valores de indice y ranking presentan una gran dispersión. Aunque la mayoría de los registros se concentra en valores relativamente bajos (la mediana del índice es 28 y la del ranking es 70), los valores máximos son extremadamente altos (64.536 para indice y 121.056 para ranking), muy alejados de sus respectivos percentiles 75 (41,23 y 110). Además, la media es considerablemente mayor que la mediana en ambas variables, lo que sugiere la presencia de valores atípicos (outliers) que influyen en el promedio. También se observa que existen datos faltantes, ya que el número de observaciones de indice (2664) y ranking (2837) es inferior al total de registros (3060).

In [14]:
#Resumen estadistico
df.describe()

#¿Qué valores mínimo, máximo y promedio tiene la columna índice?
print("Mínimo:", df["indice"].min())
print("Máximo:", df["indice"].max())
print("Promedio:", df["indice"].mean())



Mínimo: 0.0
Máximo: 64536.0
Promedio: 205.7823160660661


In [16]:
#¿Qué países presentan los valores extremos en índice y ranking?
df.loc[df["indice"].idxmin(), ["pais", "indice"]]


,1304
pais,Dinamarca
indice,0.0


In [17]:
#¿Qué países presentan los valores extremos en índice y ranking?
df.loc[df["indice"].idxmax(), ["pais", "indice"]]


,2069
pais,Kosovo
indice,64536.0


In [18]:
#¿Qué países presentan los valores extremos en índice y ranking?
df.loc[df["ranking"].idxmin(), ["pais", "ranking"]]


,53
pais,Finlandia
ranking,1.0


In [19]:
#¿Qué países presentan los valores extremos en índice y ranking?
df.loc[df["ranking"].idxmax(), ["pais", "ranking"]]

,2249
pais,Kosovo
ranking,121056.0


In [20]:
#¿Cuántos valores nulos hay en cada columna?
df.isnull().sum()

,0
codigo_iso,0
anio,0
indice,396
ranking,223
pais,0


In [21]:
#¿Qué proporción de observaciones tienen valores faltantes?
(df.isnull().sum() / len(df)) * 100

,0
codigo_iso,0.000000
anio,0.000000
indice,12.941176
ranking,7.287582
pais,0.000000


In [22]:
#¿Hay columnas con más del 30% de datos faltantes?
(df.isnull().mean() > 0.30)

,0
codigo_iso,False
anio,False
indice,False
ranking,False
pais,False


In [23]:
#¿Cuántos países distintos (pais) hay en el DataFrame?
df["pais"].nunique()

179

In [24]:
#¿Cuántos años distintos (anio) hay representados?
df["anio"].nunique()

17

In [25]:
#¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?
df.duplicated().sum()

np.int64(0)

In [26]:
#¿Hay inconsistencias entre el país y su código?
df.groupby("codigo_iso")["pais"].nunique()

,pais
codigo_iso,
AFG,1
AGO,1
ALB,1
AND,1
ARE,1
...,...
WSM,1
YEM,1
ZAF,1


**Comparación regional: países latinoamericanos.**

In [36]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df[df["codigo_iso"].isin(america)]

In [37]:
df_america = df[df["codigo_iso"].isin(america)].copy()
df_america = df_america.dropna(subset=["indice"])

A.

In [38]:
anios = sorted(df_america["anio"].unique())

for a in anios:
    df_year = df_america[df_america["anio"] == a]

    # extremos del índice
    mejor = df_year.loc[df_year["indice"].idxmin(), ["pais", "indice"]]
    peor = df_year.loc[df_year["indice"].idxmax(), ["pais", "indice"]]

    print(f"Año {a}")
    print("Mayor libertad de prensa (menor índice):")
    print(mejor.to_dict())
    print("Menor libertad de prensa (mayor índice):")
    print(peor.to_dict())
    print("-" * 50)

Año 2001
Mayor libertad de prensa (menor índice):
{'pais': 'Canadá', 'indice': 0.8}
Menor libertad de prensa (mayor índice):
{'pais': 'Cuba', 'indice': 90.3}
--------------------------------------------------
Año 2002
Mayor libertad de prensa (menor índice):
{'pais': 'Trinidad y Tobago', 'indice': 1.0}
Menor libertad de prensa (mayor índice):
{'pais': 'Cuba', 'indice': 97.83}
--------------------------------------------------
Año 2003
Mayor libertad de prensa (menor índice):
{'pais': 'Trinidad y Tobago', 'indice': 2.0}
Menor libertad de prensa (mayor índice):
{'pais': 'Argentina', 'indice': 35826.0}
--------------------------------------------------
Año 2004
Mayor libertad de prensa (menor índice):
{'pais': 'Trinidad y Tobago', 'indice': 2.0}
Menor libertad de prensa (mayor índice):
{'pais': 'Cuba', 'indice': 87.0}
--------------------------------------------------
Año 2005
Mayor libertad de prensa (menor índice):
{'pais': 'Bolivia', 'indice': 4.5}
Menor libertad de prensa (mayor índic

B.

In [39]:
mejores = df_america.loc[df_america.groupby("anio")["indice"].idxmin()]
peores = df_america.loc[df_america.groupby("anio")["indice"].idxmax()]

**Análisis anual del índice por país**

In [40]:
tabla_pivot = pd.pivot_table(
    df,
    index="pais",
    columns="anio",
    values="indice",
    aggfunc="max",
    fill_value=0
)

In [41]:
tabla_pivot

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vietnam,81.3,89.17,86.88,73.25,67.25,79.25,86.17,81.67,75.75,71.78,72.36,72.63,74.27,73.96,75.05,74.93
West Bank y Gaza,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,42.90,42.96,44.68
Yemen,34.8,41.83,48.00,46.25,54.00,56.67,59.00,83.38,82.13,69.22,67.26,66.36,67.07,65.80,62.23,61.66


**Preguntas adicionales**

a) ¿Qué país tiene el mayor valor de indice en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?

In [42]:
tabla_pivot.max().max()
tabla_pivot.stack().idxmax()

('Kosovo', np.int64(2014))

In [43]:
tabla_sin_ceros = tabla_pivot.replace(0, np.nan)
tabla_sin_ceros.min().min()
tabla_sin_ceros.stack().idxmin()

('Austria', np.int64(2009))

b) ¿Qué años presentan en promedio los valores de indice más altos? ¿Y los más bajos?

In [45]:
promedio_anual = tabla_pivot.mean(axis=0)
promedio_anual.idxmax(), promedio_anual.max()


(np.int64(2013), 449.11446927374294)

In [46]:
promedio_anual.idxmin(), promedio_anual.min()

(np.int64(2001), 20.03240223463687)

c) ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo indice a lo largo del tiempo)?

In [47]:
variabilidad = tabla_pivot.max(axis=1) - tabla_pivot.min(axis=1)

In [48]:
variabilidad.idxmax(), variabilidad.max()

('Kosovo', 64536.0)

d) ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

In [49]:
constantes = tabla_pivot.apply(lambda x: x.nunique() == 1, axis=1)

In [50]:
tabla_pivot[constantes].index.tolist()

[]

e) ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?

In [51]:
sin_datos = tabla_pivot.apply(lambda x: (x == 0).all(), axis=1)

In [52]:
tabla_pivot[sin_datos].index.tolist()

[]